In this notebook, we are using the K-Nearest Neighbors (KNN) Regressor to predict house prices based on their features using a dataset from Kaggle.<br>Geometrically, the algorithm treats every house as a data point in space, searches for the $k$ most similar houses to our query point, and averages their market values to calculate the predicted price. <br> Because raw values like square footage and bedroom counts have vastly different numerical ranges, we will first apply MinMaxScaler to normalize the dimensions. Then, we will use Feature Weighting to modify the Euclidean distance formula, mathematically stretching or shrinking specific axes so the algorithm prioritizes the features that matter most to property value.

$$d(A, B) = \sqrt{\sum_{j=1}^{d} w_j (A_j - B_j)^2}$$

In [206]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
df=pd.read_csv('data.csv')
df_cleaned = df[
    (df['sqft_living'] > 200) & (df['sqft_living'] < 8000) & 
    (df['bedrooms'] > 0) & (df['bedrooms'] < 10) &
    (df['price'] > 10000) & (df['price'] < 2000000)&
    (df['yr_built']>1800)
].dropna()

In [ ]:
data,label=df_cleaned[["bedrooms","bathrooms","sqft_living","sqft_lot","yr_built","condition","floors","view"]],df_cleaned["price"]

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(data,label,test_size=0.2,random_state=49)

In [ ]:
# scaling
scalar=MinMaxScaler()

In [ ]:
X_train_scaled=scalar.fit_transform(X_train)
X_test_scaled=scalar.transform(X_test)

In [ ]:
#Applying feature weight for selection

weight=np.array([2.0, 1.0, 3.0, 2.0, 2.0, 2.5, 2.0, 1.5])

X_train_weighted=X_train_scaled * weight
X_test_weighted=X_test_scaled * weight 

In [ ]:
# For regression, 'weights=distance' makes closer houses influence the price more than further ones
knn_housing = KNeighborsRegressor(n_neighbors=11, weights='distance', metric='euclidean')

# Fit the model on the weighted coordinate space
knn_housing.fit(X_train_weighted, y_train)

# Predict prices for the test set
y_pred = knn_housing.predict(X_test_weighted)

# Calculate error metrics
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"R² Score (Model Variance Accuracy): {r2 * 100:.2f}%")


#I was able to get r2 score of only 55. May be ,it would increase if we encode city 

In [ ]:
my_dream_house_data = {
    'bedrooms': [4],
    'bathrooms': [2],
    'sqft_living': [2200],
    'sqft_lot': [2500],
    'yr_built':[1978],
    'condition': [4],
    'floors': [1.5],
    'view': [2]
    
}

my_dream_house=pd.DataFrame(my_dream_house_data)


my_house_scaled = scalar.transform(my_dream_house)
my_house_weighted = pd.DataFrame(my_house_scaled * weight, columns=my_dream_house.columns)

# Predict the final continuous price
predicted_price = knn_housing.predict(my_house_weighted)
print(f"Estimated Market Value for this house: ${predicted_price[0]:,.2f}")

In [ ]:
df.head()